
---

# 第15章 使用RNN和CNN处理序列：

## 一、本章核心主题：处理“序列数据”
本章的核心是学习**如何处理序列数据**（文本、语音、时间序列等），重点介绍两类模型：
- **循环神经网络（RNN）**
- **卷积神经网络（CNN）** 在序列处理上的应用（如WaveNet）

---

## 二、序列数据的本质：随时间变化的信息
序列数据的特点是：
- 输入不是固定大小的图像，而是**任意长度的有序序列**（比如一句话、一段音频、股价时间序列）。
- 序列中的每个元素都和它的上下文（前后元素）相关，需要模型能记住“过去的信息”。

### 生活中的序列预测例子
- 外野手预测棒球轨迹
- 语音转文字、自动翻译
- 股票价格预测
- 自动驾驶中预测汽车行驶轨迹

---

## 三、RNN：处理序列的核心模型
### 1. 什么是RNN？
RNN（循环神经网络）是一类**专门设计来处理序列数据**的网络，它和前馈网络的最大区别是：
- 它有“循环连接”，可以把上一步的输出作为下一步的输入，从而**记住序列的历史信息**。

### 2. RNN的训练方法
- 用**时间反向传播（BPTT）**算法训练，把序列按时间步展开，反向传播误差。

### 3. RNN面临的两大核心问题
1.  **不稳定的梯度（梯度消失/爆炸）**
    - 解决方法：递归Dropout、递归层归一化等技术。
2.  **有限的短期记忆**
    - 解决方法：使用LSTM和GRU单元，让模型能记住更长时间的信息。

---

## 四、序列处理的其他方法
本章也会介绍RNN之外的方案：
- **CNN处理序列**：
  对于短序列，普通全连接网络就能处理；对于长序列（如文本、音频），CNN也能胜任。
  代表模型：**WaveNet**，用卷积网络处理音频序列，效果非常好。

---

## 五、后续章节预告
- 第16章会继续探索RNN，并介绍**注意力机制**等更先进的架构。

---

### 一句话总结
本章前言的核心是：**序列数据的本质是“有序、依赖上下文”的信息，RNN是处理这类数据的核心模型，但存在梯度和记忆问题，本章会介绍这些问题的解决方案，以及CNN处理序列的方法。**

---

# 15.1 循环神经元和层



## 一、核心概念：什么是循环神经元？
### 1. 基本结构
循环神经元（Recurrent Neuron）和普通神经元的区别，在于它多了一条**自连接的循环边**：
- 每个时间步 `t`，它同时接收两个输入：
  1. 当前输入 `xₜ`
  2. 上一个时间步的输出 `yₜ₋₁`
- 然后通过激活函数，得到当前步的输出 `yₜ`，并把它传递到下一个时间步。

### 2. 时间展开（核心理解方式）
为了便于理解和训练，我们可以把RNN在时间维度上“展开”：
- 每个时间步 `t` 对应一个“复制”的神经元，共享同一套权重。
- 信息会沿着时间轴流动，每个时间步的输出会作为下一个时间步的输入。

---

## 二、循环层的数学公式
### 1. 单个循环层的输出公式
$$ y_t = \phi(W_x^T x_t + W_y^T y_{t-1} + b) $$
- $W_x$：当前输入的权重矩阵
- $W_y$：上一时间步输出的权重矩阵
- $b$：偏置项
- $\phi$：激活函数（通常用`tanh`，而不是ReLU）

### 2. 小批量矩阵形式（更高效）
$$ Y_t = \phi(X_t W_x + Y_{t-1} W_y + b) $$
也可以写成合并形式：
$$ Y_t = \phi\left( \begin{bmatrix} X_t & Y_{t-1} \end{bmatrix} \begin{bmatrix} W_x \\ W_y \end{bmatrix} + b \right) $$

---

## 三、关键细节拆解
1.  **两组权重**
    - 输入权重 $W_x$：处理当前时间步的输入 $x_t$。
    - 循环权重 $W_y$：处理上一时间步的输出 $y_{t-1}$。
    - 两组权重在所有时间步中**共享**，这是RNN的核心设计。

2.  **初始状态**
    - 第一个时间步 `t=0` 时，没有上一步的输出 $y_{-1}$，通常初始化为全0向量。

3.  **激活函数选择**
    - RNN中通常使用 `tanh` 激活函数，而不是ReLU。
    - `tanh` 的输出范围是 `[-1, 1]`，更有利于稳定梯度传播。

---

## 四、记忆单元：RNN的“短期记忆”
### 1. 记忆的本质
循环神经元的输出 $y_t$，是由**所有过去的输入**共同决定的：
- $y_t = f(x_t, y_{t-1})$
- $y_{t-1} = f(x_{t-1}, y_{t-2})$
- ...
- 因此 $y_t$ 间接包含了 $x_0, x_1, ..., x_t$ 的信息，这就是它的“记忆”。

### 2. 基础RNN的局限
- 单个循环神经元或简单RNN层，**短期记忆能力非常有限**，通常只能记住大约10个时间步的信息。
- 对于更长的序列，它会很快“忘记”前面的信息，后续章节会介绍LSTM和GRU来解决这个问题。

---

## 五、与前馈网络的对比
| 特性 | 前馈网络 | 循环网络（RNN） |
| :--- | :--- | :--- |
| 连接方式 | 只有前向连接 | 有循环连接（输出到输入） |
| 输入长度 | 固定 | 任意长度的序列 |
| 信息传递 | 只在当前层流动 | 可以跨时间步传递信息 |
| 训练方式 | 反向传播 | 时间反向传播（BPTT） |

---

### 一句话总结
15.1节的核心是：**循环神经元通过自连接，让模型可以记住序列的历史信息，从而处理任意长度的序列数据，但简单RNN的短期记忆能力有限。**

---

## 15.1.1 记忆单元



## 15.1.1 记忆单元（Memory Cell）

## 一、记忆单元的本质
在循环神经网络中，**记忆单元（或简称单元）**就是在时间步上“保留状态”的部分，它的作用是让网络记住序列中前面的信息。

- 在时间步 $t$，单元的**隐藏状态** $h_t$，是由当前输入 $x_t$ 和上一时间步的隐藏状态 $h_{t-1}$ 共同决定的：
  $$ h_t = f(h_{t-1}, x_t) $$
- 输出 $y_t$ 也由当前状态和输入决定，在简单RNN中，$y_t = h_t$，但在更复杂的单元中，输出和状态可以不同。

---

## 二、基础RNN单元的局限
- 基础的RNN单元，短期记忆能力非常有限，通常只能记住大约 **10个时间步** 的信息。
- 对于更长的序列，比如长文本、长音频，基础RNN会很快“忘记”前面的信息，导致梯度消失/爆炸，无法学习长距离依赖。

---

## 三、后续改进方向（预告）
本章后面会介绍两种更强大的单元，来解决基础RNN的记忆问题：
1.  **LSTM（长短期记忆单元）**：通过门控机制，让模型能记住更长时间的信息（通常比基础RNN长10倍）。
2.  **GRU（门控循环单元）**：LSTM的简化版本，效果接近但计算成本更低。

---

### 一句话总结
15.1.1节的核心是：**记忆单元是RNN中负责保存和传递信息的部分，基础RNN单元的短期记忆能力有限，后续章节会用LSTM/GRU来解决这个问题。**

---

## 15.1.2 输入和输出序列


RNN的强大之处，在于它可以灵活处理**不同形式的输入和输出序列**，根据任务需求，可以分为以下4种典型模式：

---

## 一、序列到序列（Sequence-to-Sequence）
- **结构**：输入是一个序列，输出也是一个等长的序列。
- **典型应用**：时间序列预测（如股票价格预测）。
- **例子**：把过去N天的股价作为输入序列，模型输出未来N天的股价序列。

---

## 二、序列到向量（Sequence-to-Vector）
- **结构**：输入是一个序列，输出是一个固定大小的向量（单个值）。
- **典型应用**：文本分类、情感分析。
- **例子**：把电影评论的单词序列作为输入，模型输出一个情感得分（如从-1到+1）。

---

## 三、向量到序列（Vector-to-Sequence）
- **结构**：输入是一个固定向量，输出是一个序列。
- **典型应用**：图像描述生成、自动字幕。
- **例子**：把一张图片（或CNN提取的特征向量）作为输入，模型输出一段描述图片内容的文字序列。

---

## 四、编码器-解码器（Encoder-Decoder）
- **结构**：
  1.  **编码器（Encoder）**：序列到向量，把输入序列压缩成一个固定向量。
  2.  **解码器（Decoder）**：向量到序列，把这个向量解码成目标序列。
- **典型应用**：机器翻译、对话系统。
- **例子**：把中文句子作为输入，编码器将其压缩成语义向量，解码器再将向量解码成英文句子。
- **优势**：比简单的序列到序列模型效果更好，因为它可以捕捉整个输入序列的上下文信息，而不用等到输入序列结束才开始生成输出。

---

## 一句话总结
这一节的核心是：**RNN的输入和输出形式非常灵活，可以根据任务需求，组合成“序列到序列、序列到向量、向量到序列、编码器-解码器”四种模式，覆盖从预测、分类到生成的多种场景。**

---

# 15.2 训练RNN



## 15.2 训练RNN：时间反向传播（BPTT）

## 一、核心思想：把RNN按时间“展开”再反向传播
训练RNN的关键技巧，就是把它在**时间维度上展开**，然后用普通的反向传播算法训练，这个方法叫做**时间反向传播（BackPropagation Through Time, BPTT）**。

### 1. 前向传播
- 把序列按时间步 `t=0,1,2,...,T` 展开，每个时间步的RNN单元都共享同一套权重 `W` 和 `b`。
- 输入 `x₀, x₁, ..., x_T`，依次计算每个时间步的输出 `y₀, y₁, ..., y_T`。

### 2. 计算损失
- 根据任务类型，选择不同的输出计算损失：
  - 序列到序列任务：计算所有时间步的输出损失。
  - 序列到向量任务：只计算最后一个时间步的输出损失。
- 例如，图中用 `C(y₂, y₃, y₄)` 作为损失，梯度只会流过这几个时间步，不会流过前面的 `y₀, y₁`。

### 3. 反向传播更新参数
- 误差从损失计算的输出时间步，沿着展开的网络反向传播到前面的所有时间步。
- 因为所有时间步共享同一套权重 `W` 和 `b`，反向传播时会在所有时间步上求和，得到最终的梯度，用来更新参数。

---

## 二、关键特点与注意事项
1.  **参数共享**
    所有时间步的RNN单元共享同一套权重，所以梯度会在所有用到的时间步上累加，保证参数能被正确更新。

2.  **梯度消失/爆炸问题**
    序列很长时，梯度在反向传播过程中会不断累积，容易出现**梯度消失**或**梯度爆炸**问题，这是RNN训练的主要难点之一。

3.  **框架简化了所有操作**
    幸运的是，在TensorFlow/Keras中，这些复杂的展开、前向/反向传播步骤都被封装好了，你只需要正常定义模型、编译和训练即可。

---

### 一句话总结
15.2节的核心是：**通过将RNN按时间展开，用时间反向传播（BPTT）算法训练，本质上就是普通反向传播在序列上的扩展，解决了序列数据的梯度传递问题。**

---

# 15.3 预测时间序列

- 时间序列：每个时间步长一个或者多个值的序列
- 单变量时间序列：每个时间步长只有一个值（只有一个指标）
- 多变量时间序列：每个时间步长有多个值
- 典型的任务是预测未来值，这称之为预测，另一种则是填补空白：预测过去的缺失值，这成为插补

In [1]:
import keras.models
import numpy as np
# 生成时间序列
def generate_time_series(batch_size,n_steps):
    freq1,freq2,offsets1,offsets2 = np.random.rand(4,batch_size,1)
    time = np.linspace(0,1,n_steps)
    series = 0.5*np.sin((time-offsets1) * (freq1*10+10)) #wave1
    series += 0.2 *np.sin((time-offsets2) * (freq2*20+20))# +wave2
    series += 0.1 * (np.random.rand(batch_size,n_steps)-0.5) # +noise
    return series[...,np.newaxis].astype(np.float32)


In [2]:
# 创建训练集，测试集，验证集
n_steps = 50
series = generate_time_series(10000,n_steps + 1)
X_train,y_train = series[:7000,:n_steps],series[:7000,-1]
X_valid,y_valid = series[7000:9000,:n_steps],series[7000:9000,-1]
X_test,y_test = series[9000:,:n_steps],series[9000:,-1]

## 15.3.1 基准指标

In [9]:
# 最简单的基准指标是预测每个序列的最后一个值，这称之为单纯预测
import numpy as np

y_pred = X_valid[:, -1]
mse = np.mean((y_valid - y_pred) ** 2)  # 误差平方的平均值，就是MSE
print("验证集MSE:", mse)

验证集MSE: 0.021948945


In [10]:
# 另一种方法是使用全连接的网络层，由于它希望每个输入都有一个平的特征列表，因此我们需要添加一个Flatten层
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[50,1]),
    keras.layers.Dense(1)
])

C:\Users\24677\anaconda3\envs\ml\lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


## 15.3.2 实现一个简单的RNN

In [11]:
model = keras.models.Sequential([
    keras.layers.SimpleRNN(1,input_shape=[None,1])
])

C:\Users\24677\anaconda3\envs\ml\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



## 一、核心目标
用一个最简单的单层RNN，来解决前面的时间序列预测问题，看看它的效果能不能超过“直接拿最后一个点当预测值”的基准模型。

---

## 二、书上的代码与解释
### 1. 最简单的RNN模型定义
```python
model = keras.models.Sequential([
    keras.layers.SimpleRNN(1, input_shape=[None, 1])
])
```

### 2. 关键细节拆解
- `keras.layers.SimpleRNN(1)`：
  - 表示这是一个**单层RNN**，只有1个循环神经元。
- `input_shape=[None, 1]`：
  - `None`：表示模型可以接收**任意长度的序列**，这是RNN的核心优势。
  - `1`：表示每个时间步的输入只有1个特征（单变量时间序列）。
- 默认情况下，`SimpleRNN` 层**只返回最后一个时间步的输出**，正好符合我们的预测任务（用前50个点预测第51个点）。

---

## 三、模型的工作原理
1.  **初始状态**：模型的初始隐藏状态 `h₀` 被设为0。
2.  **信息传递**：
    - 第一个时间步：输入 `x₀` + 初始状态 `h₀` → 计算得到输出 `y₀`（也就是新的状态 `h₁`）。
    - 第二个时间步：输入 `x₁` + 状态 `h₁` → 计算得到输出 `y₁`（也就是新的状态 `h₂`）。
    - 这个过程会一直持续到最后一个时间步 `x₄₉`，模型只输出最后一步的结果，也就是我们要预测的 `y₅₀`。
3.  **激活函数**：默认使用 `tanh` 激活函数，这也是RNN的标准选择。

---

## 四、模型的效果与对比
- 训练配置：使用MSE损失 + Adam优化器，训练20个轮次。
- 效果：模型的MSE约为0.014，比“直接取最后一个点”的基准模型好，但还不如前面的线性模型。
- 原因：这个模型太简单了，只有3个参数，学习能力有限，还不足以捕捉序列的复杂模式。

---

## 五、补充知识点：`return_sequences=True`
- 默认情况下，`SimpleRNN` 只返回**最后一个时间步的输出**，适合我们现在的单步预测任务。
- 如果设置 `return_sequences=True`，它会返回**所有时间步的输出**，这是为了堆叠多层RNN时使用的（下一节深度RNN会用到）。

---

### 一句话总结
15.3.2节的核心是：**用一个只有1个神经元的单层RNN来做时间序列预测，它的效果比基准模型好，但还不够强，需要后续的深度RNN来提升性能。**



## 15.3.3 深度RNN

In [12]:
# 堆叠神经元
model = keras.models.Sequential([
    keras.layers.SimpleRNN(20,return_sequences=True,input_shape=[None,1]),
    keras.layers.SimpleRNN(10,return_sequences=True),
    keras.layers.SimpleRNN(1)
])

In [13]:
model = keras.models.Sequential([
    keras.layers.SimpleRNN(20,return_sequences=True,input_shape=[None,1]),
    keras.layers.SimpleRNN(20),
    keras.layers.Dense(1)
])


这一节的核心，就是**把多层RNN堆叠起来，提升模型的表达能力**，解决单层RNN性能不足的问题。

---

## 一、什么是深度RNN？
深度RNN就是**把多个循环层（如SimpleRNN、LSTM、GRU）一层一层堆叠起来**，就像搭积木一样。
- 每一层RNN都会处理上一层传来的序列信息，提取更复杂的特征
- 底层捕捉基础模式（比如波浪的起伏），高层捕捉更复杂的长期规律
- 这种结构能大幅提升模型的学习能力，让它捕捉更复杂的序列模式

---

## 二、书上的代码与关键细节
### 1. 基础的深度RNN模型
```python
# 书上的三层SimpleRNN模型
model = keras.models.Sequential([
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.SimpleRNN(20, return_sequences=True),
    keras.layers.SimpleRNN(1)
])
```

### 2. 必须搞懂的关键参数：`return_sequences=True`
这是堆叠多层RNN的核心！
- **含义**：让RNN层返回**所有时间步的输出**，而不是只返回最后一步
- **为什么要加？**
  下一层RNN需要接收**完整的序列数据**作为输入，如果不加这个参数，第一层只会返回最后一个时间步的输出（2D数据），第二层就会报错（因为它需要3D的序列数据）。
- **规则**：
  - 中间的所有循环层，都必须设置 `return_sequences=True`
  - 只有最后一层循环层，可以不设置（默认只返回最后一步）

---

## 三、模型效果与对比
- 训练配置：同样使用MSE损失 + Adam优化器，训练20个轮次
- 效果：模型的MSE降到了约 **0.003**，直接击败了之前的线性模型！
- 原因：多层结构让模型能学习更复杂的序列模式，参数数量也比线性模型多得多，学习能力大幅提升。

---

## 四、优化版本：用Dense层替代最后一层RNN
书上还提供了一个更实用的改进方案：把最后一层RNN换成普通的全连接层（Dense层），效果更好。

### 改进后的代码
```python
model = keras.models.Sequential([
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.SimpleRNN(20),  # 这里不再需要return_sequences=True
    keras.layers.Dense(1)
])
```

### 为什么这么改更好？
1.  **收敛更快**：Dense层的计算更简单，模型训练起来更快
2.  **输出更灵活**：
    - SimpleRNN层默认用`tanh`激活函数，输出范围限制在`[-1, 1]`之间
    - Dense层没有这个限制，你还可以根据任务需求，给它加上其他激活函数（比如sigmoid、relu）
3.  **效果不变**：最终的预测精度和原版模型差不多，但训练效率更高


---

## 五、一句话总结
深度RNN就是**通过堆叠多层循环层，来增强模型捕捉复杂序列模式的能力**，而`return_sequences=True`是多层RNN堆叠的关键，最后一层用Dense层收尾，能让模型训练更快、输出更灵活。

---

## 15.3.4 预测未来几个时间步长

In [14]:
# 使用已经训练好的模型，使其预测下一个值，然后再将该值加入到输入中，然后再次使用该模型预测后面的值，以此类推
series = generate_time_series(1,n_steps +10)
X_new,Y_new = series[:,:n_steps],series[:,n_steps:]
X = X_new
for step_ahead in range(10):
    y_pred_one = model.predict(X[:,step_ahead:])[:,np.newaxis,:]
    x = np.concatenate([X,y_pred_one],axis=1)

Y_pred = X[:,n_steps:]

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 517ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step


In [15]:
# 训练RNN一次输出所有的10个值
series = generate_time_series(10000,n_steps+10)
X_train,Y_train = series[:7000,:n_steps],series[:7000,-10:,0]
X_valid,Y_valid = series[7000:9000,:n_steps],series[7000:9000,-10:,0]
X_test,Y_test = series[9000:,:n_steps],series[9000:,-10:,0]

In [16]:
# 现在只需要输出层有10个单元
model = keras.models.Sequential([
    keras.layers.SimpleRNN(20,return_sequences=True,input_shape=[None,1]),
    keras.layers.SimpleRNN(20),
    keras.layers.Dense(10)
])

In [17]:
# 训练完模型后，轻松地一次预测接下来的10个值
Y_pred = model.predict(X_new)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step


In [18]:
Y_pred

array([[-0.00868552,  0.41991398,  0.82124233,  0.20746198, -0.5431034 ,
         0.13182455,  0.4010777 ,  0.25108254, -0.32797655,  0.50972027]],
      dtype=float32)

In [19]:
# 准备目标序列
Y = np.empty((10000,n_steps,10))
for step_ahead in range(1,10+1):
    Y[:,:,step_ahead-1] = series[:,step_ahead:step_ahead+n_steps,0]
Y_train = Y[:7000]
Y_valid = Y[7000:9000]
Y_test = Y[9000:]

In [20]:
# 更新的模型
model = keras.models.Sequential([
    keras.layers.SimpleRNN(20,return_sequences=True,input_shape=[None,1]),
    keras.layers.SimpleRNN(20,return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

C:\Users\24677\anaconda3\envs\ml\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [42]:
import tensorflow as tf
def last_time_step_mse(Y_true, Y_pred):
    # 取最后一个时间步，用TensorFlow的API计算均方误差
    return tf.reduce_mean(tf.square(Y_true[:, -1] - Y_pred[:, -1]))

optimizer = keras.optimizers.Adam(learning_rate=0.01)
model.compile(loss = 'mse',optimizer = optimizer,metrics = [last_time_step_mse])

# 15.4 处理长序列

## 15.4.1 应对不稳定梯度问题

这一节的核心是解决RNN训练中的**梯度消失/爆炸问题**，以及如何通过各种技术让模型稳定训练。

---

## 一、RNN为什么会出现不稳定梯度？
RNN在处理长序列时，会被“时间步”展开成一个极深的网络。
- 每一步都用同一套权重进行矩阵乘法
- 反向传播时，梯度会在链式求导中不断累积
  - 权重矩阵的特征值大于1 → **梯度爆炸**（梯度越来越大，参数更新剧烈，训练震荡）
  - 权重矩阵的特征值小于1 → **梯度消失**（梯度越来越小，前面的时间步几乎学不到东西）

这就是RNN处理长序列时，训练不稳定、记不住早期信息的根本原因。

---

## 二、应对不稳定梯度的技术详解
### 1. 基础技巧：初始化、优化器、Dropout
这些在普通深度网络里常用的技巧，对RNN也有一定帮助：
- **良好的参数初始化**：控制初始权重的范围，避免一开始就进入梯度爆炸/消失的状态
- **更合适的优化器**：比如Adam，自适应调整学习率，减少震荡
- **Dropout**：防止过拟合，提升模型泛化能力

 注意：这些方法在RNN里的效果有限，甚至可能让梯度问题更严重（比如ReLU激活函数，在RNN里反而容易让梯度爆炸）。

---

### 2. 激活函数的选择：优先用 `tanh`
- **不推荐用ReLU**：在RNN的循环乘法中，ReLU的梯度要么是0要么是1，容易导致梯度爆炸或消失
- **推荐用 `tanh`**：它的输出范围在`[-1,1]`之间，梯度的范围也更稳定，是RNN的默认激活函数
- 小技巧：如果发现训练不稳定，可以通过TensorBoard监控梯度的大小，判断是否出现了爆炸/消失

---

### 3. 梯度裁剪（Gradient Clipping）
专门用来解决**梯度爆炸**的方法：
- 原理：在反向传播时，把梯度的范数限制在一个固定的最大值（比如1.0）以内
- 实现：在优化器里设置 `clipnorm` 或 `clipvalue` 参数
- 效果：防止梯度过大导致参数更新剧烈，让训练过程更稳定

---

### 4. 批归一化（Batch Normalization）：在RNN里效果有限
- 普通前馈网络的BN，是对每个层的输出做归一化，让数据分布稳定
- 但在RNN里，不能直接在**时间步之间**使用BN，只能在**递归层之前**使用
- 原因：每个时间步的输入和隐藏状态都在变化，直接用BN会破坏序列的时序依赖关系
- 结论：在RNN中，BN只能在输入层或隐藏状态更新前使用，效果远不如在CNN里明显

---

### 5. 层归一化（Layer Normalization）：RNN的“救星”
这是RNN中效果最好的归一化技术，专门解决时序数据的分布不稳定问题：
- 原理：在每个时间步，对单个样本的特征维度进行归一化，而不是跨批次归一化
- 优势：
  1. 训练和测试时行为一致，不需要保存批次的统计信息
  2. 可以直接在每个时间步的隐藏状态更新后使用，不破坏时序依赖
  3. 对梯度稳定的效果比BN好得多
- 实现：可以自定义带层归一化的RNN单元，在每个时间步的线性计算后、激活函数前加入层归一化层

---

## 三、自定义带层归一化的RNN单元（书上代码）
```python
class LNSimpleRNNCell(keras.layers.Layer):
    def __init__(self, units, activation="tanh", **kwargs):
        super().__init__(**kwargs)
        self.state_size = units
        self.output_size = units
        self.simple_rnn_cell = keras.layers.SimpleRNNCell(units, activation=None)
        self.layer_norm = keras.layers.LayerNormalization()
        self.activation = keras.activations.get(activation)

    def call(self, inputs, states):
        outputs, new_states = self.simple_rnn_cell(inputs, states)
        norm_outputs = self.activation(self.layer_norm(outputs))
        return norm_outputs, [norm_outputs]
```
- 核心逻辑：在`SimpleRNNCell`的线性计算之后、激活函数之前，加入层归一化层
- 使用方法：通过`keras.layers.RNN`包装这个自定义单元，就能构建带层归一化的RNN层

---

## 四、一句话总结
15.4.1节的核心，就是**解决RNN训练时的梯度不稳定问题**：
- 基础技巧（初始化、优化器）效果有限
- `tanh`激活函数是默认的安全选择
- 梯度裁剪解决梯度爆炸，层归一化解决梯度消失/训练不稳定，是最有效的方法

---

In [43]:
class LNSimpleRNNCell(keras.layers.Layer):
    def __init__(self,units,activation = 'tanh',**kwargs):
        super().__init__(**kwargs)
        self.state_size = units
        self.output_size = units
        self.simple_rnn_cell = keras.layers.SimpleRNNCell(units,activation = None)
        self.layers_norm = keras.layers.LayerNormalization()
        self.activation = keras.activations.get(activation)

    def call(self,inputs,states):
        outputs,new_states = self.simple_rnn_cell(inputs,states)
        norm_outputs = self.activation(self.layers_norm(outputs))
        return norm_outputs, [outputs,new_states]

In [44]:
model = keras.models.Sequential([
    keras.layers.RNN(LNSimpleRNNCell(20),return_sequences=True,input_shape=[None,1]),
    keras.layers.RNN(LNSimpleRNNCell(20),return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

C:\Users\24677\anaconda3\envs\ml\lib\site-packages\keras\src\layers\layer.py:421: UserWarning: `build()` was called on layer 'ln_simple_rnn_cell_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
C:\Users\24677\anaconda3\envs\ml\lib\site-packages\keras\src\layers\layer.py:421: UserWarning: `build()` was called on layer 'ln_simple_rnn_cell_3', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


## 15.4.2 解决短期记忆问题

### LSTM

In [45]:
# LSTM单元
model = keras.models.Sequential([
    keras.layers.LSTM(20,return_sequences=True,input_shape=[None,1]),
    keras.layers.LSTM(20,return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

In [46]:
#或者使用通用的keras.layers.RNN层
model = keras.models.Sequential([
    keras.layers.RNN(keras.layers.LSTMCell(20),return_sequences=True,input_shape=[None,1]),
    keras.layers.RNN(keras.layers.LSTMCell(20),return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])


### 背景：短期记忆问题
- 在传统 RNN 中，数据每经过一个时间步都会丢失部分信息，导致网络难以记住序列早期的内容。
- 为了解决这一问题，引入了具有长期记忆能力的单元，其中最常见的是 **长短期记忆（LSTM）** 单元。

### LSTM 单元概述
- 由 Sepp Hochreiter 和 Jürgen Schmidhuber 于 1997 年提出，后续由多位研究者改进。
- 可以像基本 RNN 单元一样使用，但性能更好：训练收敛更快，能捕捉长期依赖关系。
- 在 Keras 中，可直接使用 `LSTM` 层，或通过 `RNN` 层配合 `LSTMCell` 实现（但直接使用 `LSTM` 层在 GPU 上有优化实现）。

### LSTM 单元的核心思想
- 将状态分为两个向量：
  - **短期状态** $ h_{(t)} $（同时也是该时间步的输出 $ y_{(t)} $）
  - **长期状态** $ c_{(t)} $
- 网络可以**学习**在长期状态中存储什么、丢弃什么、以及读取什么。

### 内部结构（门控机制）
LSTM 单元内部包含四个全连接层（作用不同）：

1. **主要层（$ g_t $）**
   - 使用 `tanh` 激活，分析当前输入 $ x_t $ 和先前短期状态 $ h_{t-1} $。
   - 它的输出不会直接送出，而是由输入门选择性地存入长期状态。

2. **遗忘门（$ f_t $）**
   - 使用 `sigmoid` 激活，输出 0~1，控制长期状态中哪些部分需要删除。

3. **输入门（$ i_t $）**
   - 使用 `sigmoid` 激活，控制主要层的输出 $ g_t $ 有多少添加到长期状态。

4. **输出门（$ o_t $）**
   - 使用 `sigmoid` 激活，控制当前时间步从长期状态中读取哪些部分，输出到短期状态 $ h_t $。

### 信息流动过程
1. 长期状态 $ c_{t-1} $ 先经过**遗忘门**（逐元素乘法），丢弃部分记忆。
2. 然后加上经过**输入门**筛选后的新记忆 $ i_t \otimes g_t $，得到新的长期状态 $ c_t $。
3. 新的长期状态 $ c_t $ 直接输出（无额外变换），同时复制一份经过 `tanh` 激活。
4. 激活后的长期状态再经由**输出门**滤波，得到短期状态 $ h_t $ 和当前输出 $ y_t $。

### 计算公式（单个实例，每个时间步）
$$
\begin{aligned}
i_t &= \sigma(W_{xi}x_t + W_{hi}h_{t-1} + b_i) \\
f_t &= \sigma(W_{xf}x_t + W_{hf}h_{t-1} + b_f) \\
o_t &= \sigma(W_{xo}x_t + W_{ho}h_{t-1} + b_o) \\
g_t &= \tanh(W_{xg}x_t + W_{hg}h_{t-1} + b_g) \\
c_t &= f_t \otimes c_{t-1} + i_t \otimes g_t \\
y_t &= h_t = o_t \otimes \tanh(c_t)
\end{aligned}
$$

- $ W_{x*}, W_{h*} $：各层与输入 $ x_t $ 或前一个短期状态 $ h_{t-1} $ 连接的权重矩阵。
- $ b_* $：偏置项（注意 TensorFlow 中将遗忘门的偏置初始化为全 1 而非全 0，以避免初始时遗忘一切）。

### 变体：窥视孔连接
- 常规 LSTM 的门控制器仅查看 $ x_t $ 和 $ h_{t-1} $。
- 窥视孔连接让门也能查看长期状态：
  - 遗忘门和输入门额外查看 $ c_{t-1} $
  - 输出门额外查看 $ c_t $
- 通常能提升性能，但并非总是如此。

### 总结
LSTM 通过精巧的门控机制，使得网络能够：
- **识别重要输入**（输入门）
- **长期保存重要信息**（遗忘门保留所需）
- **在需要时提取信息**（输出门）

这使其在时间序列、长文本、语音等任务中表现优异。

### 窥视孔连接

## 窥视孔连接（Peephole Connections）

### 基本概念
在常规 LSTM 单元中，三个门（遗忘门、输入门、输出门）的控制器只能查看当前输入 $ x_t $ 和上一个短期状态 $ h_{t-1} $。
**窥视孔连接** 是一种 LSTM 变体，允许门控制器也能够查看**长期状态**，从而获得更丰富的信息以做出更好的门控决策。

### 提出者与时间
- 由 **Felix Gers** 和 **Jérôme Schmidhuber** 于 **2000 年** 提出。

### 具体实现方式
与常规 LSTM 相比，窥视孔连接添加了以下额外的连接：

| 门 | 额外查看的长期状态           |
|---|---------------------|
| 遗忘门 | 上一个长期状态 $ c_{t-1} $ |
| 输入门 | 上一个长期状态 $ c_{t-1} $ |
| 输出门 | 当前长期状态 $ c_t $      |

> 也就是说，遗忘门和输入门在决策时可以看到“过去的长时记忆”，输出门则可以看到“更新后的长时记忆”。

### 效果
- **通常能提升性能**，使 LSTM 更有效地捕捉长期依赖。
- **但并非总是有效**，具体效果可能因任务和数据集而异。

### 与常规 LSTM 的对比
| 特性 | 常规 LSTM          | 带窥视孔连接的 LSTM              |
|---|------------------|---------------------------|
| 遗忘门输入 | $ x_t, h_{t-1} $ | $ x_t, h_{t-1}, c_{t-1} $ |
| 输入门输入 | $ x_t, h_{t-1} $ | $ x_t, h_{t-1}, c_{t-1} $ |
| 输出门输入 | $ x_t, h_{t-1} $ | $ x_t, h_{t-1}, c_t $     |

> 注：该变体在实践中不如标准 LSTM 常用，但它展示了 LSTM 家族的设计灵活性。

### GRU单元

## GRU 单元（门控循环单元）

### 概述
- **提出者**：Kyunghyun Cho 等人，于 2014 年提出（同期还提出了 Encoder-Decoder 网络）。
- **定位**：LSTM 单元的**简化版本**，在保持相当性能的前提下结构更简洁，因此日益普及。
- **性能**：多项研究表明，GRU 性能与 LSTM 相当（如 Klaus Greff 等 2015 年的论文《LSTM: A Search Space Odyssey》）。

### 与 LSTM 相比的主要简化

| LSTM 特性                                    | GRU 简化                                |
|--------------------------------------------|---------------------------------------|
| 两个状态向量：长期 $c_t$ 和短期 $h_t$               | 合并为**一个状态向量** $h_t$                   |
| 三个独立的门：遗忘门 $f_t$、输入门 $i_t$、输出门 $o_t$ | 用**单个更新门** $z_t$ 同时控制遗忘和输入；**无输出门** |
| 主要层 $g_t$ 的输入仅为 $x_t$ 和 $h_{t-1}$    | 增加**重置门** $r_t$，控制前一状态有多少进入主要层      |
| 输出为短期状态 $h_t$（经输出门滤波）                    | 每个时间步**直接输出完整状态向量** $h_t$           |

> 更新门 $z_t$ 的工作方式：当 $z_t = 1$ 时，遗忘门打开（保留旧状态），输入门关闭（忽略新输入）；当 $z_t = 0$ 时相反。这实现了“存储新记忆前需先遗忘旧记忆”的逻辑。

### 计算公式（公式 15-4）

对于单个实例在每个时间步 $t$：

\
\begin{aligned}
z_t &= \sigma(W_{zx} x_t + W_{hz} h_{t-1} + b_z) \quad &&\text{(更新门)} \\
r_t &= \sigma(W_{rx} x_t + W_{hr} h_{t-1} + b_r) \quad &&\text{(重置门)} \\
g_t &= \tanh(W_{gx} x_t + W_{hg} (r_t \otimes h_{t-1}) + b_g) \quad &&\text{(候选状态)} \\
h_t &= z_t \otimes h_{t-1} + (1 - z_t) \otimes g_t \quad &&\text{(最终状态/输出)}
\end{aligned}
\

- $\otimes$ 表示逐元素乘法。
- $W_{zx}, W_{rz}, W_{gx}$ 是输入 $x_t$ 的权重矩阵；$W_{hz}, W_{hr}, W_{hg}$ 是先前状态 $h_{t-1}$ 的权重矩阵。
- $b_z, b_r, b_g$ 为偏置项。

### Keras 中的使用
```python
# 直接使用 GRU 层
keras.layers.GRU(units, return_sequences=False, ...)

# 或通过 RNN 层配合 GRUCell
keras.layers.RNN(keras.layers.GRUCell(units), ...)
```
可以像替换 `SimpleRNN` 或 `LSTM` 一样使用 `GRU` 层。

### 与 LSTM 的简单对比

| 特性 | LSTM | GRU |
|------|------|-----|
| 状态数 | 2（$c_t, h_t$） | 1（$h_t$） |
| 门数量 | 3（遗忘、输入、输出） | 2（更新、重置） |
| 输出门 | 有 | 无 |
| 参数量 | 较多 | 较少（约 3/4） |
| 性能 | 基准 | 通常相当，有时更优 |
| 适用场景 | 通用 | 数据较小或需要更快训练时 |

### 局限性（与 LSTM 共有）
- 虽然比简单 RNN 能处理更长的序列，但**短期记忆仍然有限**，难以学习超过 100 个时间步的长期模式（如长音频、极长文本）。
- **缓解方法**：使用一维卷积层（Conv1D）对输入序列进行下采样，缩短序列长度，从而帮助 GRU 捕捉更远的依赖关系。

>  **总结**：GRU 是一种轻量化、性能强劲的门控 RNN 单元，适合在计算资源有限或需要快速迭代的场景下替代 LSTM。

## 使用一维卷积层处理序列


### 背景：RNN 长期记忆的局限
- LSTM 和 GRU 虽然比简单 RNN 能处理更长的序列，但**短期记忆仍然非常有限**，难以学习超过 100 个时间步的长期模式（如音频样本、长时间序列、长句子）。
- 一种有效的解决方法是**缩短输入序列**，例如使用一维卷积层（Conv1D）进行下采样。

### 一维卷积层（Conv1D）简介
- **类比 2D 卷积**：2D 卷积在图像上滑动多个小内核，生成 2D 特征图；1D 卷积在**序列**上滑动多个内核，为每个内核生成一维特征图。
- **功能**：每个内核学习检测一个**非常短的顺序模式**（长度 ≤ 内核大小）。
- **输出形状**：若有 10 个内核，输出为 10 个一维序列（或等效为 10 通道的一维序列）。
- **序列长度变化**：
  - `padding="same"` → 输出序列长度 = 输入序列长度。
  - `padding="valid"` 或 `stride > 1` → 输出序列比输入短（需相应调整目标序列）。

### 与 RNN 的混合使用
- 可以构建由 **Conv1D + RNN**（如 GRU/LSTM）甚至一维池化层混合的神经网络。
- **示例模型**（与之前模型相同，但以 Conv1D 开始，步长 2 下采样）：
  ```python
  model = keras.models.Sequential([
      keras.layers.Conv1D(filters=20, kernel_size=4, strides=2, padding="valid", input_shape=[None, 1]),
      keras.layers.GRU(20, return_sequences=True),
      keras.layers.GRU(20, return_sequences=True),
      keras.layers.TimeDistributed(keras.layers.Dense(10))
  ])
  model.compile(loss="mse", optimizer="adam", metrics=[last_time_step_mse])
  history = model.fit(X_train, Y_train[:, 3:], epochs=20,
                      validation_data=(X_valid, Y_valid[:, 3::2]))
  ```
  - **说明**：内核大小 `4`，步长 `2`，输出序列长度约为输入的一半（下采样）。
  - **目标调整**：因卷积层的第一个输出基于时间步 0~3，需裁剪掉目标的前 3 个时间步，并每隔一步采样一次（`Y_train[:, 3::2]`）。
  - **优势**：通过缩短序列，卷积层帮助 GRU 检测更长的模式。该模型的效果通常优于纯 RNN，甚至仅用 Conv1D（去掉循环层）也能取得很好效果。


In [47]:
model = keras.models.Sequential([
    keras.layers.Conv1D(filters=20,kernel_size=4,strides = 2,padding='valid',input_shape=[None,1]),
    keras.layers.GRU(20,return_sequences=True),
    keras.layers.GRU(20,return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

model.compile(loss = 'mse',optimizer = optimizer,metrics = [last_time_step_mse])
history = model.fit(X_train,Y_train[:,3::2],epochs=20,validation_data=(X_valid,Y_valid[:,3::2]))

Epoch 1/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - last_time_step_mse: 0.0322 - loss: 0.0402 - val_last_time_step_mse: 0.0140 - val_loss: 0.0238
Epoch 2/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - last_time_step_mse: 0.0122 - loss: 0.0220 - val_last_time_step_mse: 0.0105 - val_loss: 0.0202
Epoch 3/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - last_time_step_mse: 0.0078 - loss: 0.0176 - val_last_time_step_mse: 0.0054 - val_loss: 0.0153
Epoch 4/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - last_time_step_mse: 0.0048 - loss: 0.0146 - val_last_time_step_mse: 0.0044 - val_loss: 0.0140
Epoch 5/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - last_time_step_mse: 0.0044 - loss: 0.0136 - val_last_time_step_mse: 0.0047 - val_loss: 0.0136
Epoch 6/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - last_time_step_mse: 0.0040 - loss: 0.0129 - val_last_time_step_mse: 0.0047 - val_loss: 0.0133
Epoch 7/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - last_time_step_mse: 0.0039 - loss: 0.0125 - val_las

## WaveNet



### 概述
- **提出者**：DeepMind 的 Aaron van den Oord 等人，2016 年论文《WaveNet: A Generative Model for Raw Audio》。
- **设计目标**：处理极长序列（如原始音频，每秒数万时间步），克服 LSTM/GRU 在超长序列上的局限。
- **核心技术**：堆叠**扩张卷积**（Dilated Convolution，也称为膨胀卷积），使感受野呈指数增长。

### 扩张卷积的核心思想
- 传统卷积层一次只能看到连续几个时间步（感受野 = kernel_size）。
- 扩张卷积允许在卷积核元素之间插入“空洞”，从而在不增加参数量的情况下**指数级扩大感受野**。
- **WaveNet 的扩张策略**：每层扩张率（dilation rate）**加倍**：
  - 第 1 层：dilation = 1（相当于普通卷积，感受野 2 个时间步）
  - 第 2 层：dilation = 2（感受野 4 个时间步）
  - 第 3 层：dilation = 4（感受野 8 个时间步）
  - … 以此类推：1, 2, 4, 8, 16, 32, 64, 128, 256, 512

### 典型网络结构
- **一个堆栈**：10 个扩张卷积层（扩张率 1,2,4,…,512），相当于**一个内核大小为 1024 的超高效卷积层**，但参数更少、速度更快、表达能力更强。
- **完整 WaveNet**：堆叠 **3 个这样的堆栈**（即 30 个卷积层），以进一步增强感受野和建模能力。
- **填充策略**：使用 **因果填充（causal padding）** —— 在每个卷积层之前对输入序列的**左侧**填充相应扩张率个数的零，使得输出序列长度与输入相同，且**不引入未来信息**（适用于自回归生成任务）。

### Keras 实现示例
```python
model = keras.models.Sequential()
model.add(keras.layers.InputLayer(input_shape=None, name='input'))

# 两轮扩张率：1,2,4,8, 然后重复 1,2,4,8
for rate in (1, 2, 4, 8) * 2:
    # 因果卷积，扩张率逐步增大
    model.add(keras.layers.Conv1D(filters=20, kernel_size=2,
                                  padding="causal", activation="relu",
                                  dilation_rate=rate))
# 输出层：1x1 卷积，将通道数映射到目标维度（例如10）
model.add(keras.layers.Conv1D(filters=10, kernel_size=1))

model.compile(loss="mse", optimizer="adam")
history = model.fit(X_train, Y_train, epochs=20,
                    validation_data=(X_valid, Y_valid))
```

> **因果填充** (`padding="causal"`) 等效于在左侧填充适当数量的零，然后使用 `padding="valid"` 的普通卷积，确保不会“看到”未来的时间步。

### 性能与优势
- 在原始音频任务（文本到语音、音乐生成）上取得**最先进性能**，生成极其逼真的声音。
- 能够处理**每秒数万个时间步**的序列，这是 LSTM/GRU 难以企及的。
- 参数效率高：扩张卷积在不增加参数量的前提下获得极大感受野。

>  **扩展提示**：完整的 WaveNet 还使用了**残差连接**、**门控激活单元**（类似 GRU 的门控机制）等技巧，进一步提升性能。

### 总结
WaveNet 通过堆叠扩张率指数增长的因果卷积层，高效构建了极大感受野的卷积网络，彻底摆脱了对循环层的依赖，成为处理**超长序列**（特别是原始音频）的标杆架构。

In [48]:
model = keras.models.Sequential()
model.add(keras.layers.InputLayer(input_shape=[None,1]))
for rate in (1,2,4,8)*2:
    model.add(keras.layers.Conv1D(filters=20,kernel_size=2,padding='causal',activation='relu',dilation_rate=rate))
model.add(keras.layers.Conv1D(filters=10,kernel_size=1))
model.compile(loss = 'mse',optimizer = 'adam',metrics = [last_time_step_mse])
history = model.fit(X_train,Y_train,epochs=20,validation_data=(X_valid,Y_valid))

C:\Users\24677\anaconda3\envs\ml\lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - last_time_step_mse: 0.0533 - loss: 0.0646 - val_last_time_step_mse: 0.0234 - val_loss: 0.0357
Epoch 2/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - last_time_step_mse: 0.0201 - loss: 0.0320 - val_last_time_step_mse: 0.0189 - val_loss: 0.0299
Epoch 3/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - last_time_step_mse: 0.0165 - loss: 0.0279 - val_last_time_step_mse: 0.0160 - val_loss: 0.0270
Epoch 4/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - last_time_step_mse: 0.0143 - loss: 0.0258 - val_last_time_step_mse: 0.0135 - val_loss: 0.0247
Epoch 5/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - last_time_step_mse: 0.0128 - loss: 0.0243 - val_last_time_step_mse: 0.0122 - val_loss: 0.0239
Epoch 6/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - last_time_step_mse: 0.0120 - loss: 0.0235 - val_last_time_step_mse: 0.0114 - val_loss: 0.0230
Epoch 7/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - last_time_step_mse: 0.0113 - loss: 0.0228 - val_last_t